# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrasannaSaiS/machinelearning01-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read README.md first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule (plain words): Score pages that are visible but currently capturing zero AI referrals, prioritizing high organic volume plus healthy CTR relative to position. One reason code: `VISIBLE_HIGH_CTR_OPPORTUNITY`. Action label: `REVIEW_FOR_AI_SNIPPET`.

In [2]:
# 1) Signal checks (two signals): impressions (volume) and CTR-vs-position
# Load dataset using the repo's portable loader pattern used in other notebooks.
import os, subprocess
import pandas as pd
import numpy as np
from IPython.display import display

def load_starter_csv():
    local_candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
    ]
    for path in local_candidates:
        if os.path.exists(path):
            return pd.read_csv(path)
    repo_dir = "machinelearning01-flyrank"
    if not os.path.isdir(repo_dir):
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/PrasannaSaiS/machinelearning01-flyrank.git", repo_dir
        ], check=True)
    return pd.read_csv(f"{repo_dir}/data/raw/content_refresh_anonymized.csv")

df = load_starter_csv()
df["ai_sessions_90d"] = df.get("ai_sessions_90d", 0).fillna(0)
df["has_ai"] = (df["ai_sessions_90d"] > 0).astype(int)

def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

# Ensure CTR column exists; derive if needed
if "ctr_90d" not in df.columns:
    if "sessions_90d" in df.columns and "impressions_90d" in df.columns:
        df["ctr_90d"] = df["sessions_90d"].fillna(0) / df["impressions_90d"].replace(0, np.nan).fillna(0)
    else:
        df["ctr_90d"] = 0.0

# Ensure avg_position exists; fill missing with a large value (worse position)
if "avg_position" not in df.columns:
    df["avg_position"] = np.nan
df["avg_position"] = df["avg_position"].fillna(100.0)

# Decision surface: pages visible but currently zero-AI (the queue candidate pool)
candidates = df[(df["has_ai"] == 0) & (df["impressions_90d"].fillna(0) >= 100)].copy()
print(f"Candidate pool rows (has_ai==0 & impressions>=100): {len(candidates):,}")

# Determine content_id / client_id column fallbacks for robustness
content_col = next((c for c in ['content_id','content_hash_id','page_path','content'] if c in candidates.columns), None)
client_col = next((c for c in ['client_id','client_hash_id'] if c in candidates.columns), None)
count_col = content_col if content_col is not None else candidates.columns[0]

# ----- SIGNAL CHECK 1: Impressions (volume) bucket table -----
imp_bins = [-1, 100, 500, 1000, 5000, 20000, 1e9]
imp_labels = ['<=100','100-499','500-999','1k-4.9k','5k-19.9k','20k+']
candidates["imp_bucket"] = pd.cut(candidates["impressions_90d"].fillna(0), bins=imp_bins, labels=imp_labels)
imp_table = candidates.groupby("imp_bucket", observed=True).agg(n=(count_col,"count"), ai_rate=("has_ai","mean"))
imp_table["ai_rate_pct"] = (imp_table["ai_rate"] * 100).round(2)
print("\nSignal check 1: Impressions bucket table (n, ai_rate% on full slice)")
print(imp_table[["n","ai_rate_pct"]].to_string())

# Decide verdict for impressions vs has_ai correlation
corr_imp = candidates["impressions_90d"].fillna(0).corr(candidates["has_ai"])
if corr_imp >= 0.12:
    imp_verdict = "CONFIRMED"
elif corr_imp <= -0.12:
    imp_verdict = "OPPOSITE"
else:
    imp_verdict = "MIXED"
print(f"Impressions vs has_ai correlation: {corr_imp:+.3f}  => verdict: {imp_verdict}")

# ----- SIGNAL CHECK 2: CTR vs Position (two-way bucket table) -----
# Build buckets for avg_position and ctr
pos_bins = [0, 1, 3, 6, 20, 100]
pos_labels = ["1","2-3","4-6","7-20","21+"]
candidates["pos_bucket"] = pd.cut(candidates["avg_position"].fillna(100), bins=pos_bins, labels=pos_labels, include_lowest=True)
ctr_bins = [-1, 0.005, 0.02, 0.05, 1]
ctr_labels = ["very_low","low","medium","high"]
candidates["ctr_bucket"] = pd.cut(candidates["ctr_90d"].fillna(0), bins=ctr_bins, labels=ctr_labels)

ctr_table = (
    candidates.groupby(["pos_bucket","ctr_bucket"], observed=True)
    .agg(n=(count_col,"count"), ai_rate=("has_ai","mean"))
    .reset_index()
)
ctr_table["ai_rate_pct"] = (ctr_table["ai_rate"] * 100).round(2)
print("\nSignal check 2: CTR vs Position bucket table (n, ai_rate%)")
display(ctr_table.sort_values(["pos_bucket","ctr_bucket"]).head(40))

# Correlations: ctr and -position should both trend with has_ai if signal holds
corr_ctr = candidates["ctr_90d"].fillna(0).corr(candidates["has_ai"]) if len(candidates)>0 else 0.0
corr_pos = candidates["avg_position"].replace(0, np.nan).fillna(100).corr(candidates["has_ai"]) if len(candidates)>0 else 0.0
print(f"corr(ctr, has_ai) = {corr_ctr:+.3f}, corr(avg_position, has_ai) = {corr_pos:+.3f}")

if corr_ctr >= 0.10 and corr_pos <= -0.08:
    ctr_pos_verdict = "CONFIRMED"
elif corr_ctr < 0 and corr_pos > 0:
    ctr_pos_verdict = "OPPOSITE"
else:
    ctr_pos_verdict = "MIXED"

print(f"CTR vs Position verdict: {ctr_pos_verdict}")

# Summarize the two signal checks
print("\nSummary verdicts:")
print(f" - Impressions (volume): {imp_verdict}")
print(f" - CTR vs Position     : {ctr_pos_verdict}")


Candidate pool rows (has_ai==0 & impressions>=100): 20,207

Signal check 1: Impressions bucket table (n, ai_rate% on full slice)
               n  ai_rate_pct
imp_bucket                   
<=100         12          0.0
100-499     5173          0.0
500-999     3099          0.0
1k-4.9k     6942          0.0
5k-19.9k    3677          0.0
20k+        1304          0.0
Impressions vs has_ai correlation: +nan  => verdict: MIXED

Signal check 2: CTR vs Position bucket table (n, ai_rate%)


C:\Users\prasa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\prasa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,pos_bucket,ctr_bucket,n,ai_rate,ai_rate_pct
0,1,very_low,18,0.0,0.0
1,1,low,6,0.0,0.0
2,2-3,very_low,303,0.0,0.0
3,2-3,low,178,0.0,0.0
4,2-3,medium,15,0.0,0.0
5,2-3,high,8,0.0,0.0
6,4-6,very_low,1644,0.0,0.0
7,4-6,low,1329,0.0,0.0
8,4-6,medium,188,0.0,0.0
9,4-6,high,52,0.0,0.0


corr(ctr, has_ai) = +nan, corr(avg_position, has_ai) = +nan
CTR vs Position verdict: MIXED

Summary verdicts:
 - Impressions (volume): MIXED
 - CTR vs Position     : MIXED


C:\Users\prasa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\prasa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# Build the baseline score and write the ranked queue CSV
import os

# Re-use candidates computed above; if not present (cell run order), recompute minimal slice
try:
    candidates
except NameError:
    df = load_starter_csv()
    df["ai_sessions_90d"] = df.get("ai_sessions_90d", 0).fillna(0)
    df["has_ai"] = (df["ai_sessions_90d"] > 0).astype(int)
    if "ctr_90d" not in df.columns and "sessions_90d" in df.columns:
        df["ctr_90d"] = df["sessions_90d"].fillna(0) / df["impressions_90d"].replace(0, np.nan).fillna(0)
    df["avg_position"] = df.get("avg_position", np.nan).fillna(100)
    candidates = df[(df["has_ai"] == 0) & (df["impressions_90d"].fillna(0) >= 100)].copy()

# Feature engineering for the rule (only inference-safe fields)
candidates["vis_pct"] = percentile_rank(np.log1p(candidates["impressions_90d"].fillna(0)))
candidates["ctr_pct"] = percentile_rank(candidates["ctr_90d"].fillna(0))
# position_score: higher is better (position 1 => score close to 1)
candidates["pos_score"] = 1 - (percentile_rank(candidates["avg_position"].fillna(100)))

# Baseline score (hand-crafted, simple weights). No label-derived inputs.
candidates["baseline_score"] = (
    0.60 * candidates["vis_pct"] +
    0.30 * candidates["ctr_pct"] +
    0.10 * candidates["pos_score"]
)

def assign_reason(row):
    if row["vis_pct"] >= 0.75 and row["ctr_pct"] >= 0.5:
        return "VISIBLE_HIGH_CTR_OPPORTUNITY"
    if row["vis_pct"] >= 0.75 and row["ctr_pct"] < 0.5:
        return "VISIBLE_THIN_CONTENT"
    return "LOW_SIGNAL_MONITOR_ONLY"

candidates["reason_code"] = candidates.apply(assign_reason, axis=1)
candidates["action_label"] = "REVIEW_FOR_AI_SNIPPET"

# Columns to persist in the CSV
out_cols = [
    "client_id", "content_id", "baseline_score", "reason_code", "action_label",
    "impressions_90d", "ctr_90d", "avg_position", "word_count", "content_type", "main_intent"
]
out_df = candidates.reset_index(drop=True).sort_values("baseline_score", ascending=False)
out_df_small = out_df[[c for c in out_cols if c in out_df.columns]].copy()

os.makedirs("work/outputs", exist_ok=True)
out_path = os.path.join("work","outputs","baseline_action_score.csv")
out_df_small.to_csv(out_path, index=False)
print(f"Wrote baseline queue CSV -> {out_path}  (rows: {len(out_df_small):,})")

# Save a small JSON receipt with top metrics (useful to commit)
import json
metrics = {
    "candidate_pool": int(len(out_df_small)),
    "top1_reason": out_df_small["reason_code"].iloc[0] if len(out_df_small)>0 else None,
}
with open(os.path.join("work","outputs","baseline_action_score_metrics.json"), "w") as f:
    json.dump(metrics, f)
print("Saved metrics to work/outputs/baseline_action_score_metrics.json")


Wrote baseline queue CSV -> work\outputs\baseline_action_score.csv  (rows: 20,207)
Saved metrics to work/outputs/baseline_action_score_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Below we produce a top-10 review (concise lines) as required by the assignment.

In [4]:
# Top-10 review: for each top-10 row produce (action, why, what_would_make_it_wrong)
top_k = 10
top = out_df.sort_values("baseline_score", ascending=False).head(top_k).copy()

def review_row(row):
    action = row.get("action_label", "REVIEW_FOR_AI_SNIPPET")
    reason = row.get("reason_code", "LOW_SIGNAL_MONITOR_ONLY")
    why = (
        f"Score={row['baseline_score']:.3f}; vis={row.get('impressions_90d',0):,} impressions; "
        f"ctr={row.get('ctr_90d',0):.3f}; pos={row.get('avg_position',None)}"
    )
    # what would make it wrong: list plausible failures
    problems = []
    if row.get('impressions_90d',0) < 200:
        problems.append('Impressions lower than snapshot filter (sensitive to small counts)')
    problems.append('Recent edits since snapshot changed content structure')
    if reason == 'VISIBLE_HIGH_CTR_OPPORTUNITY':
        problems.append('CTR signal driven by navigation or search pages (not helpful for snippet)')
    if row.get('word_count', np.nan) < 200:
        problems.append('Too short to contain an authoritative answer block')
    what_wrong = "; ".join(problems)
    return pd.Series({
        "client_id": row.get("client_id"),
        "content_id": row.get("content_id"),
        "action": action,
        "reason": reason,
        "why": why,
        "what_would_make_it_wrong": what_wrong,
    })

reviews = top.apply(review_row, axis=1)
print("Top-10 review (one line per page):")
for i, r in reviews.reset_index(drop=True).iterrows():
    print(f"{i+1:2d}. content_id={r['content_id']} | action={r['action']} | reason={r['reason']}")
    print(f"    why: {r['why']}")
    print(f"    what would make it wrong: {r['what_would_make_it_wrong']}\n")

# Also display a compact DataFrame if running interactively
display(reviews)


Top-10 review (one line per page):
 1. content_id=content_a05d66a41827 | action=REVIEW_FOR_AI_SNIPPET | reason=VISIBLE_HIGH_CTR_OPPORTUNITY
    why: Score=0.945; vis=57,078 impressions; ctr=0.032; pos=5.1
    what would make it wrong: Recent edits since snapshot changed content structure; CTR signal driven by navigation or search pages (not helpful for snippet)

 2. content_id=content_34e549c30fa0 | action=REVIEW_FOR_AI_SNIPPET | reason=VISIBLE_HIGH_CTR_OPPORTUNITY
    why: Score=0.924; vis=44,204 impressions; ctr=0.027; pos=6.3
    what would make it wrong: Recent edits since snapshot changed content structure; CTR signal driven by navigation or search pages (not helpful for snippet)

 3. content_id=content_a22b7f6c73c5 | action=REVIEW_FOR_AI_SNIPPET | reason=VISIBLE_HIGH_CTR_OPPORTUNITY
    why: Score=0.918; vis=28,192 impressions; ctr=0.051; pos=9.1
    what would make it wrong: Recent edits since snapshot changed content structure; CTR signal driven by navigation or search pages (n

,client_id,content_id,action,reason,why,what_would_make_it_wrong
8138,client_6208ef0f77,content_a05d66a41827,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.945; vis=57,078 impressions; ctr=0.032...",Recent edits since snapshot changed content st...
7364,client_6208ef0f77,content_34e549c30fa0,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.924; vis=44,204 impressions; ctr=0.027...",Recent edits since snapshot changed content st...
17241,client_7f2253d7e2,content_a22b7f6c73c5,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.918; vis=28,192 impressions; ctr=0.051...",Recent edits since snapshot changed content st...
20007,client_b4944c6ff0,content_eca19e020f91,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.914; vis=25,978 impressions; ctr=0.023...",Recent edits since snapshot changed content st...
4134,client_19581e27de,content_ef93da00bc37,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.911; vis=36,917 impressions; ctr=0.021...",Recent edits since snapshot changed content st...
11498,client_6208ef0f77,content_9f4a8828ea6f,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.906; vis=24,064 impressions; ctr=0.032...",Recent edits since snapshot changed content st...
494,client_19581e27de,content_227efcce3f1a,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.903; vis=36,255 impressions; ctr=0.019...",Recent edits since snapshot changed content st...
3507,client_b4944c6ff0,content_251ab03c2530,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.903; vis=79,731 impressions; ctr=0.014...",Recent edits since snapshot changed content st...
3745,client_19581e27de,content_7a5be8f2ada6,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.902; vis=11,241 impressions; ctr=0.052...",Recent edits since snapshot changed content st...
2380,client_19581e27de,content_d6b17274474e,REVIEW_FOR_AI_SNIPPET,VISIBLE_HIGH_CTR_OPPORTUNITY,"Score=0.901; vis=105,273 impressions; ctr=0.01...",Recent edits since snapshot changed content st...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# Weak picks + leakage checks
# 1) Simple weak-pick audit: top 50 rows where reason_code == VISIBLE_THIN_CONTENT (likely wrong)
weak = out_df[out_df["reason_code"] == "VISIBLE_THIN_CONTENT"].head(20)
print(f"Weak picks (VISIBLE_THIN_CONTENT) sample count: {len(weak)}")
if len(weak) > 0:
    display(weak[[c for c in ['client_id','content_id','impressions_90d','ctr_90d','word_count','baseline_score'] if c in weak.columns]].head(10))
else:
    print("No VISIBLE_THIN_CONTENT picks in top population sample.")

# 2) Leakage check: ensure we did not use outcome/label-derived columns in features
leakage_fields = ["ai_sessions_90d","trend_direction","trend_pct"]
used_fields = ['impressions_90d','ctr_90d','avg_position','word_count']
found_leak = [f for f in leakage_fields if f in used_fields and f in candidates.columns]
print("\nLeakage check:")
if found_leak:
    print("LEAKAGE FOUND: the following excluded fields were (accidentally) used:", found_leak)
else:
    print("No excluded label-derived fields used in the baseline features. PASS.")

# 3) Candidate pool re-check: confirm candidate rows have ai_sessions_90d == 0
violations = candidates[candidates['ai_sessions_90d'] > 0]
print(f"Candidate rows with ai_sessions_90d > 0 (should be 0): {len(violations)}")
if len(violations) > 0:
    display(violations.head())
else:
    print("Candidate pool correctness: PASS (no positive ai_sessions in pool).")


Weak picks (VISIBLE_THIN_CONTENT) sample count: 20


,client_id,content_id,impressions_90d,ctr_90d,word_count,baseline_score
2219,client_19581e27de,content_4fc39a2b8cf0,160959,0.007064,2907.0,0.846348
9545,client_19581e27de,content_44e481c8f55b,312694,0.006278,3040.0,0.836354
3001,client_bbb965ab0c,content_dd1d885704ea,73651,0.006857,2820.0,0.834577
1503,client_19581e27de,content_b6d2061fcd11,99523,0.006823,NaN,0.832212
20005,client_4e07408562,content_57596b8c69ff,65880,0.007089,1737.0,0.830361
476,client_349c41201b,content_a296f4390de7,41088,0.006863,3212.0,0.828547
9153,client_19581e27de,content_2c2606c5d176,347399,0.006177,NaN,0.828248
1296,client_3fdba35f04,content_91a44d04fe19,34721,0.006883,2880.0,0.826988
7908,client_19581e27de,content_153db95b47da,62118,0.006375,NaN,0.825224
17284,client_19581e27de,content_f8dac969c272,56259,0.006559,NaN,0.824853



Leakage check:
No excluded label-derived fields used in the baseline features. PASS.
Candidate rows with ai_sessions_90d > 0 (should be 0): 0
Candidate pool correctness: PASS (no positive ai_sessions in pool).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under notebooks — then submit your repo URL on the card. Done.